In [ ]:
SELECT DISTINCT
    CONCAT('SONE', s.id_organisation_source) AS cprod_src_sys_inst_id,
    CONCAT(
        'SONE',
        s.id_organisation_source,
        '_',
        LOWER(CONCAT(TRIM(s.rota_slot_type), '_', TRIM(r.rota_type)))
    ) AS generated_cprod_src_id,
    LOWER(CONCAT(TRIM(s.rota_slot_type), '_', TRIM(r.rota_type))) AS generated_cprod_name,
    s.id_rota,
    s.id_organisation_source,
    s.rota_slot_type,
    r.rota_type
FROM silver.silver_sone_srrotaslot s
LEFT JOIN silver.silver_sone_srrota r
    ON s.id_rota = r.id
WHERE s.id IS NOT NULL
  AND s.id_organisation_source IS NOT NULL
  AND s.rota_slot_type IS NOT NULL
  AND TRIM(s.rota_slot_type) <> ''
  AND r.rota_type IS NOT NULL
  AND TRIM(r.rota_type) <> ''
  AND s.blocked_slot = false
ORDER BY generated_cprod_src_id;

In [ ]:
WITH sone_generated AS (
    SELECT DISTINCT
        CONCAT('SONE', s.id_organisation_source) AS cprod_src_sys_inst_id,
        CONCAT(
            'SONE',
            s.id_organisation_source,
            '_',
            LOWER(CONCAT(TRIM(s.rota_slot_type), '_', TRIM(r.rota_type)))
        ) AS generated_cprod_src_id
    FROM silver.silver_sone_srrotaslot s
    LEFT JOIN silver.silver_sone_srrota r
        ON s.id_rota = r.id
    WHERE s.id IS NOT NULL
      AND s.id_organisation_source IS NOT NULL
      AND s.rota_slot_type IS NOT NULL
      AND TRIM(s.rota_slot_type) <> ''
      AND r.rota_type IS NOT NULL
      AND TRIM(r.rota_type) <> ''
      AND s.blocked_slot = false
)
SELECT
    g.cprod_src_sys_inst_id,
    g.generated_cprod_src_id,
    rdm.cprod_src_id,
    rdm.cprod_id
FROM sone_generated g
LEFT JOIN silver_rdm_care_product rdm
    ON LOWER(TRIM(rdm.cprod_src_id)) = LOWER(TRIM(g.generated_cprod_src_id))
   AND LOWER(TRIM(rdm.cprod_src_sys_inst_id)) = LOWER(TRIM(g.cprod_src_sys_inst_id))
WHERE rdm.cprod_id IS NULL
ORDER BY g.generated_cprod_src_id;

In [ ]:
SELECT
    id,
    COUNT(DISTINCT id_organisation_source) AS org_count,
    COUNT(*) AS row_count
FROM silver.silver_sone_srrota
GROUP BY id
HAVING COUNT(DISTINCT id_organisation_source) > 1
ORDER BY org_count DESC, row_count DESC;

In [ ]:
SELECT
    z_src_system_instance,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN session_cprod_id IS NOT NULL THEN 1 ELSE 0 END) AS cprod_id_populated_count,
    SUM(CASE WHEN session_cprod_id IS NULL THEN 1 ELSE 0 END) AS cprod_id_null_count
FROM silver_session_wip_test
WHERE z_src_system_instance LIKE 'SONE%'
GROUP BY z_src_system_instance
ORDER BY z_src_system_instance;